In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.read_csv("../data/raw/telco_data.csv")

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['tenure'] = df['tenure'].fillna(df['tenure'].median())
df['MonthlyCharges'] = df['MonthlyCharges'].fillna(df['MonthlyCharges'].median())
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
df['gender'] = df['gender'].fillna(df['gender'].mode()[0])
df['Partner'] = df['Partner'].fillna(df['Partner'].mode()[0])
df['InternetService'] = df['InternetService'].fillna(df['InternetService'].mode()[0])
df['StreamingTV'] = df['StreamingTV'].fillna(df['StreamingTV'].mode()[0])

df['charge_ratio'] = df['MonthlyCharges'] / (df['TotalCharges'] / df['tenure'].replace(0, 1))
df['tenure_group'] = pd.cut(df['tenure'], bins=[0, 12, 24, 48, 72], labels=['0-1yr', '1-2yr', '2-4yr', '4-6yr'])
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df['total_services'] = (df[service_cols] == 'Yes').sum(axis=1)
df['charges_difference'] = df['TotalCharges'] - (df['tenure'] * df['MonthlyCharges'])

df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
categorical_cols.remove('customerID')

le = LabelEncoder()
df[categorical_cols] = df[categorical_cols].apply(lambda col: le.fit_transform(col.astype(str)))

print("df rebuilt:", df.shape)

X_ltv = df.drop(columns=['customerID', 'Churn', 'TotalCharges'])
y_ltv = df['TotalCharges']
X_ltv_train, X_ltv_test, y_ltv_train, y_ltv_test = train_test_split(X_ltv, y_ltv, test_size=0.2, random_state=42)

ltv_model = RandomForestRegressor(n_estimators=100, random_state=42)
ltv_model.fit(X_ltv_train, y_ltv_train)
y_ltv_pred = ltv_model.predict(X_ltv_test)

mae = mean_absolute_error(y_ltv_test, y_ltv_pred)
r2 = r2_score(y_ltv_test, y_ltv_pred)
print(f"Mean Absolute Error: ${mae:.2f}")
print(f"R² Score: {r2:.3f}")

C:\Users\zaidm\AppData\Local\Temp\ipykernel_31988\2948294006.py:25: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()


df rebuilt: (7043, 25)
Mean Absolute Error: $73.05
R² Score: 0.996
